# HPML HW2: Profiling Small LLM Workloads
**COMS 6998 — High Performance Machine Learning, Spring 2026**  
Instructor: Dr. Kaoutar El Maghraoui  
Wei Alexander Xin (wax1)  

Fine-tuning DistilBERT on IMDB sentiment classification with profiling and hyperparameter analysis.

## Environment Setup

In [ ]:
!pip install -q datasets transformers wandb tensorboard

In [ ]:
# Standard library imports used across the notebook
import os
import platform
import time
from tqdm import tqdm

In [ ]:
# Core stack for this HW
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import AdamW, SGD, Adam
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import load_dataset
import wandb
import transformers as tf_lib

In [ ]:
# Environment info (for my sanity as much as it is required)
print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {tf_lib.__version__}")
print(f"CUDA:         {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

# Prefer GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:       {device}")

if device.type == "cuda":
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory:   {props.total_mem / 1e9:.1f} GB")
    print(f"Driver:       {torch.cuda.get_arch_list()}")

In [ ]:
# One-time W&B auth
wandb.login()

## Data Loading & Tokenization

In [ ]:
# Load IMDB sentiment data
dataset = load_dataset("imdb")
print(f"\nTrain: {len(dataset['train'])}")
print(f"Test: {len(dataset['test'])}\n")

# Shared constants for all sections
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

# Tokenizer maps raw text to model-ready token IDs
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_func(batch):
    # Truncate to keep memory and compute predictable
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

# Batched tokenization is faster than item-by-item
tokenized = dataset.map(tokenize_func, batched=True, batch_size=1000)

# Return torch tensors directly from the dataset
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Quick length sanity check
sample_lens = [len(tokenized["train"][i]["input_ids"]) for i in range(100)]
print(
    f"Token lengths (first 100): min={min(sample_lens)}, "
    f"max={max(sample_lens)}, avg={sum(sample_lens)/len(sample_lens):.0f}"
)

# Coding Exercises (100 pts total)

---
## C1: Fine-tuning a Small LLM (15 pts)

Fine-tune DistilBERT on IMDB for 5 epochs. Report training loss and accuracy per epoch.

In [ ]:
def dynamic_padding(batch):
    """Pad each batch to the longest sequence in that batch."""
    # Saves compute versus padding every sample to MAX_LEN
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])
    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask, batch_first=True, padding_value=0
    )
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

def move_batch_to_device(batch, device):
    """Move a collated batch to CPU/GPU with optional async transfer."""
    # non_blocking helps overlap host->device copies on CUDA
    non_blocking = device.type == "cuda"
    input_ids = batch["input_ids"].to(device, non_blocking=non_blocking)
    attention_mask = batch["attention_mask"].to(device, non_blocking=non_blocking)
    labels = batch["labels"].to(device, non_blocking=non_blocking)
    return input_ids, attention_mask, labels

def run_train_step(model, optimizer, input_ids, attention_mask, labels):
    """Forward + backward + optimizer update for a single batch."""
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    outputs.loss.backward()
    optimizer.step()
    return outputs

def update_classification_counts(logits, labels):
    """Return (# correct, # total) for a logits/labels pair."""
    preds = logits.argmax(dim=-1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total

def train_one_epoch(model, dataloader, optimizer, device, desc="Training"):
    """Run one full training epoch and return (avg_loss, avg_accuracy)."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(dataloader, desc=desc):
        input_ids, attention_mask, labels = move_batch_to_device(batch, device)
        outputs = run_train_step(model, optimizer, input_ids, attention_mask, labels)
        batch_correct, batch_total = update_classification_counts(outputs.logits, labels)

        # Handles short final batches with weighted loss
        total_loss += outputs.loss.item() * batch_total
        correct += batch_correct
        total += batch_total

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, dataloader, device):
    """Run evaluation pass and return accuracy."""
    model.eval()
    correct = 0
    total = 0

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids, attention_mask, labels = move_batch_to_device(batch, device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        batch_correct, batch_total = update_classification_counts(outputs.logits, labels)
        correct += batch_correct
        total += batch_total

    return correct / total

In [ ]:
# Baseline defaults reused in multiple sections
DEFAULT_BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 2
DEFAULT_LR = 1e-4
DEFAULT_EPOCHS = 5
DEFAULT_OPTIMIZER = "AdamW"

def build_run_config(
    batch_size=DEFAULT_BATCH_SIZE,
    lr=DEFAULT_LR,
    optimizer_name=DEFAULT_OPTIMIZER,
    num_workers=DEFAULT_NUM_WORKERS,
    epochs=DEFAULT_EPOCHS,
    compile_mode=False,
):
    """Create a consistent W&B config dictionary for all experiments."""
    return {
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "batch_size": batch_size,
        "lr": lr,
        "optimizer": optimizer_name,
        "num_workers": num_workers,
        "epochs": epochs,
        "compile_mode": compile_mode,
    }

def create_dataloaders(batch_size=DEFAULT_BATCH_SIZE, num_workers=DEFAULT_NUM_WORKERS):
    """Factory for train/test dataloaders with shared collate and pin-memory policy."""
    loader_kwargs = {
        "collate_fn": dynamic_padding,
        # pin_memory can improve transfer speed on CUDA
        "pin_memory": torch.cuda.is_available(),
        "num_workers": num_workers,
    }
    train_loader = DataLoader(
        tokenized["train"],
        batch_size=batch_size,
        shuffle=True,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        tokenized["test"],
        batch_size=batch_size,
        shuffle=False,
        **loader_kwargs,
    )
    return train_loader, test_loader

def create_optimizer(model, lr=DEFAULT_LR, optimizer_name=DEFAULT_OPTIMIZER):
    """Factory for optimizer selection used in C1/C6 experiments."""
    opt_map = {"AdamW": AdamW, "Adam": Adam, "SGD": SGD}
    return opt_map[optimizer_name](model.parameters(), lr=lr)

def create_model(lr=DEFAULT_LR, optimizer_name=DEFAULT_OPTIMIZER):
    """Build model + optimizer pair so every run starts from scratch."""
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(device)
    optimizer = create_optimizer(model, lr=lr, optimizer_name=optimizer_name)
    return model, optimizer

# Baseline loaders and model for C1/C2
train_loader, test_loader = create_dataloaders()
model, optimizer = create_model()

# Parameter counts are useful for checks and short answers
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Batches: {len(train_loader)}")

In [ ]:
# C1 baseline run
c1_config = build_run_config(epochs=DEFAULT_EPOCHS)
wandb.init(project="hpml-hw2-llm", name="c1-baseline", config=c1_config)

for epoch in range(DEFAULT_EPOCHS):
    epoch_num = epoch + 1
    t0 = time.perf_counter()

    loss, acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device,
        f"Epoch {epoch_num}/{DEFAULT_EPOCHS}",
    )
    test_acc = evaluate(model, test_loader, device)

    wandb.log(
        {
            "epoch": epoch_num,
            "train/loss": loss,
            "train/acc": acc,
            "test/acc": test_acc,
        }
    )

    print(
        f"Epoch {epoch_num}/{DEFAULT_EPOCHS} | Loss: {loss:.4f} | "
        f"Train Acc: {acc:.4f} | Test Acc: {test_acc:.4f} | "
        f"Time: {time.perf_counter() - t0:.1f}s"
    )

wandb.finish()

---
## C2: Baseline Timing (10 pts)

For each epoch, measure and report:  
a) Data loading time  
b) Training computation time (forward + backward + optimizer)  
c) Total epoch time  

Uses `torch.cuda.synchronize()` for accurate GPU timing. Log to W&B.

In [ ]:
def get_synced_time(device):
    """Return wall-clock time after synchronizing GPU work when needed."""
    # GPU kernels are async, so sync before timing
    if device.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter()

def get_epoch_duration(t_start, device):
    """Standardized total epoch time measurement."""
    return get_synced_time(device) - t_start

def train_one_epoch_timed(model, dataloader, optimizer, device, desc="Training"):
    """Train one epoch and return (loss, acc, data_time, compute_time)."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    data_time = 0.0
    compute_time = 0.0

    # Tracks previous loop end time
    t_end = get_synced_time(device)

    for batch in tqdm(dataloader, desc=desc):
        # DataLoader wait time
        t_data = get_synced_time(device)
        data_time += t_data - t_end

        input_ids, attention_mask, labels = move_batch_to_device(batch, device)
        outputs = run_train_step(model, optimizer, input_ids, attention_mask, labels)

        # Times forward + backward + optimizer step
        t_compute = get_synced_time(device)
        compute_time += t_compute - t_data

        batch_correct, batch_total = update_classification_counts(outputs.logits, labels)
        total_loss += outputs.loss.item() * batch_total
        correct += batch_correct
        total += batch_total
        t_end = t_compute

    return total_loss / total, correct / total, data_time, compute_time

In [ ]:
# Fresh model for C2 timing
model, optimizer = create_model()

c2_config = build_run_config(epochs=DEFAULT_EPOCHS)
wandb.init(project="hpml-hw2-llm", name="c2-baseline-timed", config=c2_config)

for epoch in range(DEFAULT_EPOCHS):
    epoch_num = epoch + 1
    t0 = time.perf_counter()

    loss, acc, dt, ct = train_one_epoch_timed(
        model,
        train_loader,
        optimizer,
        device,
        f"Epoch {epoch_num}/{DEFAULT_EPOCHS}",
    )
    test_acc = evaluate(model, test_loader, device)
    et = get_epoch_duration(t0, device)

    wandb.log(
        {
            "epoch": epoch_num,
            "train/loss": loss,
            "train/acc": acc,
            "test/acc": test_acc,
            "time/data_loading": dt,
            "time/compute": ct,
            "time/epoch": et,
        }
    )

    print(
        f"Epoch {epoch_num}/{DEFAULT_EPOCHS} | Loss: {loss:.4f} | "
        f"Train Acc: {acc:.4f} | Test Acc: {test_acc:.4f}"
    )
    print(f"  Data: {dt:.2f}s | Compute: {ct:.2f}s | Total: {et:.2f}s")

wandb.finish()

---
## C3: DataLoader Performance (10 pts)

Vary `num_workers` in {0, 2, 4, 8}. For each setting, report total data loading time and plot _epoch
time vs. num_workers_. Identify the optimal number of workers and briefly discuss. Log all runs to
W&B and create a W&B Table comparing the configurations.

In [ ]:
def log_epoch_metrics(epoch, loss, acc, test_acc, data_time, compute_time, epoch_time):
    """Single place for per-epoch W&B metric logging."""
    wandb.log(
        {
            "epoch": epoch,
            "train/loss": loss,
            "train/acc": acc,
            "test/acc": test_acc,
            "time/data_loading": data_time,
            "time/compute": compute_time,
            "time/epoch": epoch_time,
        }
    )

def print_epoch_summary(epoch, num_epochs, loss, acc, test_acc, data_time, compute_time, epoch_time):
    """Consistent console summary for each experiment epoch."""
    print(
        f"Epoch {epoch}/{num_epochs} | Loss: {loss:.4f} | "
        f"Train Acc: {acc:.4f} | Test Acc: {test_acc:.4f}"
    )
    print(f"  Data: {data_time:.2f}s | Compute: {compute_time:.2f}s | Total: {epoch_time:.2f}s")

def run_experiment(
    run_name,
    num_epochs=DEFAULT_EPOCHS,
    batch_size=DEFAULT_BATCH_SIZE,
    lr=DEFAULT_LR,
    optimizer_name=DEFAULT_OPTIMIZER,
    num_workers=DEFAULT_NUM_WORKERS,
    compile_mode=False,
):
    """Run a full train/eval experiment and return aggregate metrics."""
    config = build_run_config(
        batch_size=batch_size,
        lr=lr,
        optimizer_name=optimizer_name,
        num_workers=num_workers,
        epochs=num_epochs,
        compile_mode=compile_mode,
    )

    # Resets model state for fair comparisons
    model, optimizer = create_model(lr=lr, optimizer_name=optimizer_name)
    if compile_mode:
        # compile usually pays setup cost before speeding up
        model = torch.compile(model, backend="inductor")

    train_loader, test_loader = create_dataloaders(
        batch_size=batch_size,
        num_workers=num_workers,
    )

    wandb.init(project="hpml-hw2-llm", name=run_name, config=config)

    epoch_results = []
    for epoch_idx in range(num_epochs):
        epoch_num = epoch_idx + 1
        t0 = time.perf_counter()

        loss, acc, dt, ct = train_one_epoch_timed(
            model,
            train_loader,
            optimizer,
            device,
            f"Epoch {epoch_num}/{num_epochs}",
        )
        test_acc = evaluate(model, test_loader, device)
        et = get_epoch_duration(t0, device)

        log_epoch_metrics(epoch_num, loss, acc, test_acc, dt, ct, et)
        print_epoch_summary(epoch_num, num_epochs, loss, acc, test_acc, dt, ct, et)
        epoch_results.append(
            {
                "train_loss": loss,
                "train_acc": acc,
                "test_acc": test_acc,
                "data_time": dt,
                "compute_time": ct,
                "epoch_time": et,
            }
        )

    wandb.finish()

    total_time = sum(e["epoch_time"] for e in epoch_results)
    total_data_time = sum(e["data_time"] for e in epoch_results)
    return {
        "config": config,
        "epochs": epoch_results,
        "final_train_loss": epoch_results[-1]["train_loss"],
        "final_train_acc": epoch_results[-1]["train_acc"],
        "final_test_acc": epoch_results[-1]["test_acc"],
        "total_time": total_time,
        "avg_epoch_time": total_time / num_epochs,
        "total_data_time": total_data_time,
    }

def log_wandb_table(run_name, table_key, columns, rows):
    """Utility for C3/C5/C6 comparison tables."""
    wandb.init(project="hpml-hw2-llm", name=run_name)
    table = wandb.Table(columns=columns)
    for row in rows:
        table.add_data(*row)
    wandb.log({table_key: table})
    wandb.finish()

In [ ]:
# C3 worker sweep
c3_results = []
for nw in [0, 2, 4, 8]:
    print(f"\n{'='*60}\nnum_workers={nw}\n{'='*60}")
    result = run_experiment(f"c3-nw{nw}", num_workers=nw)
    result["num_workers"] = nw
    c3_results.append(result)

In [ ]:
# C3 summary plots
nw_vals = [r["num_workers"] for r in c3_results]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, ylabel, title in [
    (
        ax1,
        [r["avg_epoch_time"] for r in c3_results],
        "Avg Epoch Time (s)",
        "Epoch Time vs. num_workers",
    ),
    (
        ax2,
        [r["total_data_time"] for r in c3_results],
        "Total Data Loading Time (s)",
        "Data Loading Time vs. num_workers",
    ),
]:
    ax.bar(range(len(nw_vals)), vals, tick_label=[str(n) for n in nw_vals])
    ax.set_xlabel("num_workers")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    # Labels bars for quick screenshot reading
    for i, v in enumerate(vals):
        ax.text(i, v + 0.5, f"{v:.1f}s", ha="center")

plt.tight_layout()
plt.show()

# Picks best worker count for this setup
OPTIMAL_NW = min(c3_results, key=lambda r: r["avg_epoch_time"])["num_workers"]
print(f"\nOptimal num_workers = {OPTIMAL_NW}")

In [ ]:
# C3 comparison table for W&B
c3_rows = [
    (
        r["num_workers"],
        round(r["total_data_time"], 2),
        round(r["avg_epoch_time"], 2),
        round(r["final_test_acc"], 4),
    )
    for r in c3_results
]

log_wandb_table(
    run_name="c3-summary",
    table_key="c3_num_workers_comparison",
    columns=["num_workers", "total_data_time (s)", "avg_epoch_time (s)", "final_test_acc"],
    rows=c3_rows,
)

### C3 Discussion

*TODO: Fill in after running*

---
## C4: PyTorch Profiler (15 pts)

Use `torch.profiler` with TensorBoard to generate profiler traces and visualizations.  
**Important:** You do **not** need to profile the entire 5-epoch run. Instead:  
* Run the profiler for a representative slice of training (e.g., one epoch or a few hundred batches).
* The goal is to capture a realistic breakdown of time spent in data loading, forward, backward, and optimizer.
* A single full epoch is usually enough. If profiler traces get too large or slow, limit to a smaller subset — just make sure the results are representative.  

Compare runs with `num_workers=1` and the optimal value from C3.  

#### Required deliverables:  
a) **TensorBoard Profiler Screenshots** showing the **Overview** tab for both `num_workers` configurations. Your screenshots must include:
* GPU Summary (utilization, memory usage, efficiency metrics)
* Execution Summary table (showing time breakdown by category: Kernel, Memcpy, DataLoader, etc.)
* Execution Summary pie chart (visual breakdown of time percentages)
* Step Time Breakdown (stacked bar chart showing categories across steps)
* Performance Recommendation section (if available)  

b) **Comparison table** showing the percentage of time spent in each category (DataLoader, Kernel/GPU computation, CPU Exec, Memcpy, etc.) for both num_workers configurations.  
c) **Analysis** (2-3 paragraphs) discussing:
* How does increasing num_workers affect the DataLoader percentage?
& What percentage of time is spent on GPU computation (Kernel) vs. data loading?
* Are there any bottlenecks identified by the profiler?
* What do the performance recommendations suggest (if any)?  

**Implementation hint:** Use `torch.profiler.profile` with `tensorboard_trace_handler` and view results in TensorBoard’s PyTorch Profiler plugin. See https://pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html for reference.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity, schedule, tensorboard_trace_handler

def run_profile_steps(model, optimizer, dataloader, desc, max_steps, prof=None, annotate=False):
    """Run a bounded number of train steps, optionally with profiler annotations."""
    model.train()
    for step, batch in enumerate(tqdm(dataloader, desc=desc)):
        input_ids, attention_mask, labels = move_batch_to_device(batch, device)
        optimizer.zero_grad()

        if annotate:
            # Named regions appear in the profiler timeline
            with record_function("forward"):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
            with record_function("backward"):
                outputs.loss.backward()
            with record_function("optimizer_step"):
                optimizer.step()
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            outputs.loss.backward()
            optimizer.step()

        if prof is not None:
            prof.step()

        if step >= max_steps:
            # Keep trace size manageable
            break

# C4 asks for num_workers=1 vs best from C3
for nw in [1, OPTIMAL_NW]:
    print(f"\n{'=' * 60}\nProfiling num_workers={nw}\n{'=' * 60}")
    log_dir = f"./profiler_logs/nw{nw}"
    prof_loader, _ = create_dataloaders(batch_size=32, num_workers=nw)
    model, optimizer = create_model()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=schedule(skip_first=5, wait=1, warmup=2, active=6, repeat=2),
        on_trace_ready=tensorboard_trace_handler(log_dir),
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    ) as prof:
        run_profile_steps(
            model,
            optimizer,
            prof_loader,
            desc=f"Profiling nw={nw}",
            max_steps=50,
            prof=prof,
            annotate=True,
        )

    print(f"Traces saved to {log_dir}/")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./profiler_logs

### C4 Screenshots

*TODO: Screenshots of TensorBoard Overview tab (GPU Summary, Execution Summary, Step Time Breakdown, Recommendations)*

### C4 Comparison Table

| Category | num_workers=1 (%) | num_workers=OPTIMAL (%) |
|----------|-------------------|-------------------------|
| Kernel (GPU compute) | | |
| Memcpy | | |
| CPU Exec | | |
| DataLoader | | |
| Other | | |

### C4 Analysis

*TODO: 2-3 paragraphs*

---
## C5: Hyperparameter Sensitivity (15 pts)

Sweep:  
* **Batch size:** 16, 32, 64
* **Learning rate:** $5 \times 10^{-5}$, $1 \times 10^{-4}$, $5 \times 10^{-4}$

For each configuration, report final training loss/accuracy, test accuracy, and total training time. Discuss trade-offs among speed, accuracy, and stability. Log all runs to W&B and create a W&B Table summarizing the results.

In [ ]:
# C5 hyperparameter sweep
c5_results = []
for bs in [16, 32, 64]:
    for lr in [5e-5, 1e-4, 5e-4]:
        print(f"\n{'='*60}\nbatch_size={bs}, lr={lr}\n{'='*60}")
        result = run_experiment(
            f"c5-bs{bs}_lr{lr}",
            batch_size=bs,
            lr=lr,
            num_workers=OPTIMAL_NW,
        )
        result["batch_size"] = bs
        result["lr"] = lr
        c5_results.append(result)

In [ ]:
# Console summary table
print(f"{'BS':>4} | {'LR':>8} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Acc':>8} | {'Time (s)':>8}")
print("-" * 65)
for r in c5_results:
    print(
        f"{r['batch_size']:>4} | {r['lr']:>8.0e} | {r['final_train_loss']:>10.4f} | "
        f"{r['final_train_acc']:>9.4f} | {r['final_test_acc']:>8.4f} | {r['total_time']:>8.1f}"
    )

# Rounds values for easier table scanning
c5_rows = [
    (
        r["batch_size"],
        r["lr"],
        round(r["final_train_loss"], 4),
        round(r["final_train_acc"], 4),
        round(r["final_test_acc"], 4),
        round(r["total_time"], 1),
    )
    for r in c5_results
]

log_wandb_table(
    run_name="c5-summary",
    table_key="c5_hyperparam_comparison",
    columns=["batch_size", "lr", "final_train_loss", "final_train_acc", "final_test_acc", "total_time (s)"],
    rows=c5_rows,
)

### C5 Discussion

*TODO: Discuss trade-offs among speed, accuracy, and stability*

---
## C6: Optimizer Comparison (10 pts)

Train 5 epochs each with SGD, Adam, and AdamW.

In [ ]:
# C6 optimizer comparison
c6_results = []
for opt_name in ["SGD", "Adam", "AdamW"]:
    print(f"\n{'='*60}\nOptimizer: {opt_name}\n{'='*60}")
    result = run_experiment(
        f"c6-{opt_name.lower()}",
        optimizer_name=opt_name,
        num_workers=OPTIMAL_NW,
    )
    result["optimizer"] = opt_name
    c6_results.append(result)

In [ ]:
print(f"{'Optimizer':>10} | {'Avg Epoch (s)':>13} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Acc':>8}")
print("-" * 65)
for r in c6_results:
    print(
        f"{r['optimizer']:>10} | {r['avg_epoch_time']:>13.2f} | {r['final_train_loss']:>10.4f} | "
        f"{r['final_train_acc']:>9.4f} | {r['final_test_acc']:>8.4f}"
    )

c6_rows = [
    (
        r["optimizer"],
        round(r["avg_epoch_time"], 2),
        round(r["final_train_loss"], 4),
        round(r["final_train_acc"], 4),
        round(r["final_test_acc"], 4),
    )
    for r in c6_results
]

log_wandb_table(
    run_name="c6-summary",
    table_key="c6_optimizer_comparison",
    columns=["optimizer", "avg_epoch_time (s)", "final_train_loss", "final_train_acc", "final_test_acc"],
    rows=c6_rows,
)

### C6 Discussion

*TODO: Compare performance and convergence behavior*

---
## C7: torch.compile (10 pts)

Compare eager mode vs. `torch.compile(model, backend="inductor")`. 10 epochs each.

In [ ]:
# C7 eager vs compile (Inductor)
# First compiled epoch is slower from setup overhead
c7_eager = run_experiment("c7-eager", num_epochs=10, num_workers=OPTIMAL_NW)
c7_compiled = run_experiment("c7-compiled", num_epochs=10, num_workers=OPTIMAL_NW, compile_mode=True)

In [ ]:
# HW asks for first epoch and average of epochs 6-10
eager_first = c7_eager["epochs"][0]["epoch_time"]
eager_avg = sum(e["epoch_time"] for e in c7_eager["epochs"][5:]) / 5
compiled_first = c7_compiled["epochs"][0]["epoch_time"]
compiled_avg = sum(e["epoch_time"] for e in c7_compiled["epochs"][5:]) / 5

print(f"{'Mode':<20} | {'First Epoch (s)':>15} | {'Avg Epochs 6-10 (s)':>20}")
print("-" * 62)
print(f"{'Eager':<20} | {eager_first:>15.2f} | {eager_avg:>20.2f}")
print(f"{'Compile (Inductor)':<20} | {compiled_first:>15.2f} | {compiled_avg:>20.2f}")

# Ratio > 1 means compile is faster after warmup
print(f"\nSpeedup (epochs 6-10): {eager_avg / compiled_avg:.2f}x")

---
## C8: Advanced Profiling (Extra Credit, 10 pts)

In [ ]:
# C8 operator-level profile on a representative subset
prof_loader, _ = create_dataloaders(batch_size=32, num_workers=OPTIMAL_NW)
model, optimizer = create_model()

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    run_profile_steps(
        model,
        optimizer,
        prof_loader,
        desc="Profiling (operator-level)",
        max_steps=20,
        prof=prof,
        annotate=False,
    )

# Top-k tables are easy to cite in analysis
print("\nTop 20 CUDA operators by total time:")
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))
print("\nTop 20 CPU operators by total time:")
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

### C8 Analysis

*TODO: Identify at least 2 bottlenecks and propose concrete optimizations*

---
# Short Answer Questions (15 pts)

### Q1 (3 pts): Input dimension of DistilBERT's embedding layer?

The vocabulary size: **30,522**. Each token ID indexes into the embedding matrix to produce a 768-dim vector.

### Q2 (3 pts): Output dimension of the classifier head for IMDB?

**2** — one logit per class (negative, positive).

### Q3 (5 pts): Trainable parameters and parameters with gradients

In [ ]:
# Q3 parameter counts and gradient coverage
model, _ = create_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

# Uses a tiny batch to trigger backward quickly
# Quick check for parameters with gradients
dummy = tokenizer("test", return_tensors="pt", padding="max_length", max_length=16, truncation=True)
dummy = {k: v.to(device) for k, v in dummy.items()}
model(**dummy, labels=torch.tensor([0], device=device)).loss.backward()

with_grads = sum(p.numel() for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients after backward: {with_grads:,}")

### Q4 (2 pts): Does parameter count change switching SGD to Adam?

**No.** Parameter count is determined by architecture. Adam adds optimizer state (momentum + variance) which uses more *memory*, but these aren't model parameters.

### Q5 (2 pts): Why is first epoch slower with torch.compile?

**First epoch:** `torch.compile` traces the graph and generates optimized Triton/C++ kernels. One-time compilation cost.

**Later epochs:** cached compiled graph with operator fusion — fewer kernel launches, less Python overhead, reduced memory traffic.